# AI Lab - Practical Question No. 27
## Greedy Best-First Search Grid Navigator (Informed Search & Heuristics)

| Specification | Detail |
| :--- | :--- |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🟠 Advanced |
| **Practical Type** | Problem Solving |
| **Core AI Concept** | Informed Search limitations, Pure Heuristic $f(n) = h(n)$ |
| **Python Concepts Tested** | Dynamic heuristic calculation, min-heap priority queue via `heapq` |
| **Complexity** | $\mathcal{O}(N \log N)$ |
| **Sample Input** | Start: (0,0), Goal: (9,9) on 10x10 Maze |
| **Sample Output** | `Expanded: 25 nodes, Path Length: 22 (Suboptimal)` |
| **Viva Topics** | Greedy vs $A^*$, Influence of obstacles on Greedy Search |
| **Optional Challenge** | Output the grid with the path marked by `*` |

## 1. Theoretical Foundations & Evaluation Function

### Pure Heuristic Evaluation: $f(n) = h(n)$
In **Greedy Best-First Search (GBFS)**, the agent explores states evaluated solely by an estimate of the remaining distance to the goal:
$$f(n) = h(n)$$
Unlike **Uniform Cost Search** ($f(n) = g(n)$) or **$A^*$ Search** ($f(n) = g(n) + h(n)$), Greedy Best-First Search completely ignores the accumulated step cost $g(n)$ from the start node.

### Dynamic Manhattan Distance Heuristic
For 4-directional grid navigation (Up, Down, Left, Right), the natural heuristic is the **Manhattan Distance** ($L_1$ norm):
$$h_M((r, c), (r_{\text{goal}}, c_{\text{goal}})) = |r - r_{\text{goal}}| + |c - c_{\text{goal}}|$$
Manhattan distance is dynamically calculated based on the destination goal coordinates at runtime.

### Fundamental Trade-Off: Efficiency vs. Optimality
- **Fewer Expansions**: Guided by $h(n)$, GBFS focuses exploration directly toward the goal like a beam, expanding significantly fewer nodes than blind search algorithms like BFS.
- **Suboptimal Solutions**: Because GBFS ignores $g(n)$, it eagerly pursues locally attractive steps that can lead to obstacle detours or dead-end traps, resulting in suboptimal (longer) paths.

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.gbfs_grid_navigator import (
    DEFAULT_10X10_MAZE,
    DEFAULT_START_COORD,
    DEFAULT_GOAL_COORD,
    CONCAVE_TRAP_10X10_MAZE,
    OPEN_10X10_MAZE,
    UNSOLVABLE_10X10_MAZE,
    GBFSGridNavigator,
    greedy_best_first_search_grid,
    breadth_first_search_grid,
    a_star_search_grid,
    compare_gbfs_vs_bfs,
    render_grid_with_path,
    render_side_by_side_grids,
    manhattan_distance,
    euclidean_distance,
    chebyshev_distance,
    get_gbfs_grid_viva_reference,
)

print("Libraries and Question 27 modules successfully imported!")

## 2. Canonical Syllabus Sample Execution

Let us run Greedy Best-First Search on the canonical 10x10 maze from Start `(0, 0)` to Goal `(9, 9)`.
We verify that it outputs the exact expected sample output:
```
Expanded: 25 nodes, Path Length: 22 (Suboptimal)
```

In [ ]:
# Execute Greedy Best-First Search on 10x10 maze
gbfs_result = greedy_best_first_search_grid(
    DEFAULT_10X10_MAZE,
    DEFAULT_START_COORD,
    DEFAULT_GOAL_COORD,
    record_steps=True,
)

# Check suboptimality relative to BFS optimal baseline
comp = compare_gbfs_vs_bfs(DEFAULT_10X10_MAZE, DEFAULT_START_COORD, DEFAULT_GOAL_COORD)
gbfs_result.is_suboptimal = comp.is_gbfs_suboptimal

print("=== Canonical Syllabus Sample Output ===")
print(gbfs_result.format_sample_output())
print(f"\nExact Path Coordinates ({len(gbfs_result.path)} coordinates, {gbfs_result.path_length} steps):")
print(gbfs_result.path)

## 3. Comparative Analysis: Greedy Best-First Search vs. Breadth-First Search (BFS)

Here we demonstrate the core AI principle requested in the problem statement:
> **"The program must demonstrate that Greedy search explores fewer nodes than BFS but might take a suboptimal path."**

- **Nodes Expanded**: Greedy explores **25 nodes**, while BFS must explore **87 nodes** (Greedy achieves a **71.3% reduction** in search effort).
- **Path Length**: BFS guarantees the optimal shortest path of **18 steps**, while Greedy takes a **22-step path** (Suboptimal detour).

In [ ]:
# Run comprehensive comparative benchmark
comparison = compare_gbfs_vs_bfs(DEFAULT_10X10_MAZE, DEFAULT_START_COORD, DEFAULT_GOAL_COORD)
print(comparison.format_summary_table())

## 4. Optional Challenge: Rendering Grid with Path Marked by `*`

The syllabus provides an **Optional Challenge**: *"Output the grid with the path marked by '*'"*.

Legend:
- `S`: Start location `(0, 0)`
- `G`: Goal location `(9, 9)`
- `*`: Steps traversed by Greedy Best-First Search
- `#`: Impassable wall / obstacle
- `.`: Open walkable cell

In [ ]:
# Optional Challenge: Render 10x10 maze with path marked by '*'
print("=== Optional Challenge: Path Marked Grid ('*') ===")
print(render_grid_with_path(DEFAULT_10X10_MAZE, gbfs_result.path, start=DEFAULT_START_COORD, goal=DEFAULT_GOAL_COORD))

print("\n=== Side-by-Side Path Comparison (GBFS '*' vs BFS 'o') ===")
print(render_side_by_side_grids(DEFAULT_10X10_MAZE, comparison.gbfs_result, comparison.bfs_result))

## 5. Influence of Obstacles & Concave Traps (Local Minima)

When obstacles form a concave U-shaped trap or cul-de-sac, Greedy Search plunges directly inside because $h(n)$ decreases towards the back wall. Once stuck at the local minimum, it must backtrack out.

In [ ]:
# Demonstrate Concave Trap (U-shaped obstacle)
trap_result = greedy_best_first_search_grid(CONCAVE_TRAP_10X10_MAZE, (0, 0), (9, 9), record_steps=True)
print(f"Concave Trap Result: Nodes Expanded={trap_result.nodes_expanded}, Path Length={trap_result.path_length}")
print("\nRendered Trap Grid:")
print(render_grid_with_path(CONCAVE_TRAP_10X10_MAZE, trap_result.path, start=(0,0), goal=(9,9)))

## 6. $A^*$ Search: The Optimal Informed Solution

By incorporating $g(n)$, **$A^*$ Search** ($f(n) = g(n) + h(n)$) achieves the best of both worlds: it utilizes heuristic guidance while guaranteeing mathematical path optimality.

In [ ]:
astar_result = a_star_search_grid(DEFAULT_10X10_MAZE, DEFAULT_START_COORD, DEFAULT_GOAL_COORD)
print(f"A* Search -> Path Length: {astar_result.path_length} (Guaranteed Optimal), Nodes Expanded: {astar_result.nodes_expanded}")
print(f"GBFS     -> Path Length: {gbfs_result.path_length} (Suboptimal), Nodes Expanded: {gbfs_result.nodes_expanded}")
print(f"BFS      -> Path Length: {comparison.bfs_result.path_length} (Optimal), Nodes Expanded: {comparison.bfs_result.nodes_expanded}")

## 7. University Viva Voce Examination Guide

Essential questions and model answers covering:
- **Greedy vs $A^*$ Search**
- **Influence of obstacles on Greedy Search**
- **Admissibility of Manhattan distance on grids**

In [ ]:
viva = get_gbfs_grid_viva_reference()
print(f"=== {viva['title']} ===\n")
for idx, qa in enumerate(viva["qa_pairs"], start=1):
    print(f"Q{idx}: {qa['question']}")
    print(f"Ans: {qa['answer']}\n")